# LoRA fine-tune: teach the model which candidate is the width

### Why this, and why now

Three zero-shot rounds established that a 2B VLM does **not** resolve
depth/width/height, but they also pinned down *how* it fails. On rows where the
correct value is provably among the options offered, it scored **43.3%** against
a uniform-random **33.7%** — barely above chance — and its choices were badly
skewed:

```
(b) 36.9%   (c) 35.4%   (a) 11.1%   (d) 8.1%   (e-h) 8.5%
```

With 3.48 options on average, an unbiased selector spreads ~28% across (a)–(d).
**It is answering from positional habit, not from the image.** That is a prior,
and priors are what supervision overwrites.

Two design choices follow directly from that diagnosis:

1. **Train on the selection task** — output a letter, not a measurement. Clean
   labels (weak supervision already knows which candidate matches gold) and it
   targets the failing decision exactly.
2. **Shuffle the option order every epoch.** If the model can score well only by
   learning "usually (b)", shuffling destroys that shortcut and forces it to use
   the image. This is the single most important line in the notebook.

| Reference (398 held-out rows) | dim |
|---|---|
| VLM zero-shot, open-ended | 11.0% |
| VLM zero-shot, forced | 35.4% |
| VLM zero-shot, select | 34.2% |
| tagger + rules cascade | **36.2%** ← the bar |
| gold among candidates | 70.1% ← hard ceiling |

The 70.1% is a real cap: it is how often OCR surfaces a qualifying candidate at
all. Nothing trained this way can exceed it.

### Setup

- **Accelerator → `GPU T4 x2`**, **Internet → On**
- **Add-ons → Secrets** → add `HF_TOKEN` (raises rate limits; needed to push the
  adapter)
- **Add Input** → attach the dataset holding `vlm_lora_data.tar.gz`

`MODEL_ID` in the config cell switches to a 7B — but run 2B first. It is ~1 h
against many hours, and it is what tests the diagnosis.

In [ ]:
import glob, os, subprocess, sys, tarfile

# ---- config -------------------------------------------------------------
MODEL_ID   = "Qwen/Qwen2-VL-2B-Instruct"   # -> "Qwen/Qwen2.5-VL-7B-Instruct" only if 2B plateaus
EPOCHS     = 1          # one pass answers the actual question (do the letters
                        # flatten?) inside a single 12 h Kaggle session
BATCH      = 1          # visual tokens are the memory cost, not parameters
ACCUM      = 8          # effective batch 8
LR         = 1e-4
MAX_PIXELS = 512 * 512  # ~335 visual tokens vs ~750 at 768*768: about half the
                        # compute per step. The model picks among candidates OCR
                        # already read, so it needs layout, not fine print.
GRAD_CKPT  = "auto"     # "auto": probe the heaviest example and turn checkpointing
                        # on only if it would not fit. It costs ~20-30% of step time.
LORA_R     = 16
SEED       = 7
LOAD_4BIT  = False      # set True for a 7B on a single T4
# -------------------------------------------------------------------------

found = (glob.glob("/kaggle/input/**/train.csv", recursive=True)
         + glob.glob("**/train.csv", recursive=True))
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
if found:
    DEST = os.path.dirname(os.path.abspath(found[0]))
else:
    tars = (glob.glob("/kaggle/input/**/vlm_lora_data.tar.gz", recursive=True)
            + glob.glob("**/vlm_lora_data.tar.gz", recursive=True))
    if not tars:
        raise SystemExit("No train.csv or vlm_lora_data.tar.gz under /kaggle/input.")
    with tarfile.open(tars[0]) as t:
        t.extractall(WORK)
    DEST = os.path.join(WORK, "vlm_lora")
print("data:", DEST, "|", len(os.listdir(f"{DEST}/images")), "images")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("no HF_TOKEN secret (fine for public models):", type(e).__name__)

# Set before torch is imported or it has no effect. Both spellings: the name
# changed to PYTORCH_ALLOC_CONF in recent torch, and the old one still works.
for _k in ("PYTORCH_CUDA_ALLOC_CONF", "PYTORCH_ALLOC_CONF"):
    os.environ[_k] = "expandable_segments:True"

import torch
print("cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("Enable GPU: Settings panel > Accelerator > GPU T4 x2")

In [ ]:
# ---- dependency repair -------------------------------------------------
# Read versions from package METADATA, never by importing. Kaggle's image ships
# torchao 0.10.0; a transformers new enough to want torchao>=0.16.0 raises
# ImportError the moment `import transformers` runs, so an import-based version
# check cannot survive to report the problem it was added to detect.
from importlib.metadata import PackageNotFoundError, version as _v
import packaging.version as V

def ver(pkg):
    try:
        return _v(pkg)
    except PackageNotFoundError:
        return None

tv, tao, pv = ver("transformers"), ver("torchao"), ver("peft")
print("before:  transformers", tv, "| torchao", tao, "| peft", pv)

# Nothing here uses TorchAO quantization (we are bf16/fp16, or bitsandbytes for
# 4-bit). Removing it is strictly cheaper than upgrading a CUDA-compiled package
# and it dissolves the version conflict outright.
if tao and tv and V.parse(tao) < V.parse("0.16.0") and V.parse(tv) >= V.parse("4.50"):
    print("torchao", tao, "is too old for transformers", tv, "-> uninstalling (unused)")
    subprocess.run([sys.executable, "-m", "pip", "-q", "uninstall", "-y", "torchao"], check=False)

pkgs = []
if tv is None or V.parse(tv) < V.parse("4.45"):
    pkgs.append("transformers>=4.45")       # Qwen2-VL support landed in 4.45
if pv is None:
    # --no-deps on purpose: a plain `-U peft` pulls a newer transformers with it,
    # which is what broke this notebook the first time.
    print("installing peft (--no-deps; accelerate/safetensors already present)")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "--no-deps", "peft"], check=True)
if LOAD_4BIT and ver("bitsandbytes") is None:
    pkgs.append("bitsandbytes")
if pkgs:
    print("installing:", pkgs)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *pkgs], check=True)

print("after:   transformers", ver("transformers"), "| torchao", ver("torchao"), "| peft", ver("peft"))

# Prove the repair before the expensive cells depend on it.
import transformers
from transformers import Qwen2VLForConditionalGeneration  # noqa: F401
import peft  # noqa: F401
print("imports OK - transformers", transformers.__version__, "| peft", peft.__version__)


In [ ]:
import csv, json, math, random, re, string, time
from collections import Counter, defaultdict

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, DEST)
import normalize as N

PHRASE = {"depth": "depth (front to back)",
          "width": "width (left to right)",
          "height": "height (top to bottom)"}
SELECT = ("This product photograph shows measurements. Text recognition read "
          "these values from the image:\n{options}\n"
          "Which one is the product's {phrase}?\n"
          "Answer with only the letter.")

def parse_cands(s):
    out = []
    for part in (s or "").split(";"):
        m = re.match(r"\s*(-?\d+(?:\.\d+)?)\s+(\S+)\s*$", part)
        if m:
            u = N.normalize_unit(m.group(2))
            if u:
                out.append((float(m.group(1)), u))
    return out

def load(name):
    rows = []
    for r in csv.DictReader(open(f"{DEST}/{name}.csv")):
        r["cands"] = parse_cands(r["candidates"])
        r["gold_index"] = int(r["gold_index"])
        rows.append(r)
    return rows

train_rows, eval_rows = load("train"), load("eval")
print(f"train {len(train_rows):,} | eval {len(eval_rows):,}")
print("train attribute mix:", dict(Counter(r["attribute"] for r in train_rows)))

def render(row, order):
    """Prompt + correct letter under a given option ordering."""
    opts = "\n".join(f"  ({string.ascii_lowercase[j]}) {row['cands'][k][0]:g} {row['cands'][k][1]}"
                      for j, k in enumerate(order))
    prompt = SELECT.format(options=opts, phrase=PHRASE[row["attribute"]])
    target = None
    if row["gold_index"] >= 0 and row["gold_index"] in order:
        target = string.ascii_lowercase[order.index(row["gold_index"])]
    return prompt, target

print("\nexample:\n" + render(train_rows[0], list(range(len(train_rows[0]["cands"]))))[0])

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import LoraConfig, get_peft_model

proc = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=MAX_PIXELS)

kw = dict(dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
          attn_implementation="sdpa")   # eager attention materialises the full
                                        # attention matrix; with ~750 visual tokens
                                        # that alone can cost a GB per layer
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=kw["dtype"],
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    kw["device_map"] = "auto"

base = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, **kw)
if not LOAD_4BIT:
    base = base.to("cuda")

# Adapt the language tower's attention and MLP projections. The vision encoder
# stays frozen: the task is not "see better", it is "stop answering from a
# positional prior", which lives in the decoder.
lora = LoraConfig(
    r=LORA_R, lora_alpha=LORA_R*2, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(base, lora)
model.print_trainable_parameters()
model.config.use_cache = False

# Gradient checkpointing. Without it, every layer's activations are kept alive
# for the backward pass. At 768*768 (~750 visual tokens through 28 layers) that
# put 13.6 GiB on a 14.6 GiB T4. Checkpointing recomputes them in backward
# instead: roughly 20-30% slower, several GiB cheaper. At 512*512 it may not be
# needed at all, which is what the probe in the next cell finds out.
def set_ckpt(on):
    if on:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    else:
        model.gradient_checkpointing_disable()
    print("gradient checkpointing:", "ON" if on else "off")

set_ckpt(GRAD_CKPT is True)   # "auto" is resolved by a probe step in the next cell
# The embedding output must require grad for checkpointing to have a graph to
# rebuild - the trainable LoRA weights all sit downstream of it. Harmless when off.
model.enable_input_require_grads()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
print(f"weights on GPU: {torch.cuda.memory_allocated()/2**30:.2f} GiB "
      f"of {torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GiB")

In [ ]:
class SelectDS(Dataset):
    """Shuffles the option order on every __getitem__.

    This is the point of the whole exercise: the zero-shot model answered (b) or
    (c) 72% of the time. If the ordering is random per epoch, that shortcut
    earns nothing and the image is the only remaining signal.
    """
    def __init__(self, rows, seed=SEED, shuffle_options=True):
        self.rows = [r for r in rows if len(r["cands"]) >= 2 and r["gold_index"] >= 0]
        self.rng = random.Random(seed)
        self.shuffle_options = shuffle_options

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        order = list(range(len(row["cands"])))
        if self.shuffle_options:
            self.rng.shuffle(order)
        prompt, target = render(row, order)
        return row["image_file"], prompt, target

def collate(batch):
    texts, images, targets = [], [], []
    for fname, prompt, target in batch:
        msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
        chat = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        texts.append(chat + target)
        images.append(Image.open(f"{DEST}/images/{fname}").convert("RGB"))
        targets.append(target)

    enc = proc(text=texts, images=images, return_tensors="pt", padding=True)
    labels = enc["input_ids"].clone()
    labels[labels == proc.tokenizer.pad_token_id] = -100
    # Supervise the final answer token only. Training on the prompt too would
    # spend capacity re-learning the candidate list we already handed over.
    for i in range(labels.shape[0]):
        keep = (labels[i] != -100).nonzero().flatten()
        if len(keep):
            labels[i][keep[:-1]] = -100
    enc["labels"] = labels
    return enc

ds = SelectDS(train_rows)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
print(f"trainable examples: {len(ds):,}  ->  {len(dl)//ACCUM*EPOCHS} optimiser steps")

if GRAD_CKPT == "auto":
    # One forward+backward on the longest prompt (most candidates), without
    # checkpointing. Keep 15% headroom for images whose aspect ratio yields a
    # few more visual tokens than the probe's.
    heavy = max(range(len(ds)), key=lambda i: len(ds.rows[i]["cands"]))
    total_mem = torch.cuda.get_device_properties(0).total_memory
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    model.train()
    try:
        b = {k: v.to("cuda") for k, v in collate([ds[heavy]]).items()}
        model(**b).loss.backward()
        peak = torch.cuda.max_memory_allocated()
        fits = peak < 0.85 * total_mem
        print(f"probe without checkpointing: peak {peak/2**30:.2f} GiB "
              f"of {total_mem/2**30:.2f} GiB ({b['input_ids'].shape[1]} tokens)")
    except torch.cuda.OutOfMemoryError:
        fits = False
        print("probe without checkpointing: OOM")
    finally:
        model.zero_grad(set_to_none=True)
        b = None
        torch.cuda.empty_cache()
    set_ckpt(not fits)
    torch.cuda.reset_peak_memory_stats()

In [ ]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
total = max(1, len(dl)//ACCUM*EPOCHS)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=total,
                                            pct_start=0.05, anneal_strategy="linear")
model.train()
step, run, skipped, t0 = 0, 0.0, 0, time.time()
for epoch in range(EPOCHS):
    for i, batch in enumerate(dl):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        try:
            loss = model(**batch).loss / ACCUM
            loss.backward()
            run += loss.item()
        except torch.cuda.OutOfMemoryError:
            # One unusually wide image should cost us that row, not the run.
            skipped += 1
            opt.zero_grad(set_to_none=True)
            del batch
            torch.cuda.empty_cache()
            continue
        if (i+1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % 20 == 0:
                el = time.time()-t0
                print(f"  epoch {epoch+1} step {step}/{total}  loss {run/20*ACCUM:.4f}  "
                      f"{el/60:.1f} min elapsed  eta {(total-step)*el/step/60:.0f} min  "
                      f"peak {torch.cuda.max_memory_allocated()/2**30:.1f} GiB", flush=True)
                run = 0.0
ADAPTER = f"{WORK}/qwen2vl-dim-lora"
model.save_pretrained(ADAPTER)
print("\nsaved adapter ->", ADAPTER, f"({(time.time()-t0)/60:.1f} min)")
print(f"peak GPU {torch.cuda.max_memory_allocated()/2**30:.2f} GiB | rows skipped on OOM: {skipped}")

# Hand the memory back before generation: optimiser state and gradients are
# dead weight during eval, and generate() wants room for a KV cache.
opt.zero_grad(set_to_none=True)
del opt
torch.cuda.empty_cache()

In [ ]:
# ---- evaluate on the held-out 398, with options in their natural OCR order ----
model.eval()
model.gradient_checkpointing_disable()   # recompute is pointless without a backward
model.config.use_cache = True

def ask(fname, prompt):
    msgs = [{"role":"user","content":[{"type":"image"},{"type":"text","text":prompt}]}]
    chat = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    img = Image.open(f"{DEST}/images/{fname}").convert("RGB")
    inp = proc(text=[chat], images=[img], return_tensors="pt").to("cuda")
    with torch.no_grad():
        gen = model.generate(**inp, max_new_tokens=6, do_sample=False)
    return proc.batch_decode(gen[:, inp["input_ids"].shape[1]:],
                             skip_special_tokens=True)[0].strip()

def dim_ok(pred, lo, hi, gu, tol=0.01):
    if pred is None: return False
    v, u = pred
    if N.family(u) != N.family(gu): return False
    a = N.to_base(v, u)
    return N.to_base(lo, gu)*(1-tol) <= a <= N.to_base(hi, gu)*(1+tol)

out, letters = [], Counter()
t0 = time.time()
for i, r in enumerate(eval_rows):
    cands = r["cands"]
    if not cands:
        pred, raw = None, "(no candidates)"
    elif len(cands) == 1:
        pred, raw = cands[0], "(single)"
    else:
        order = list(range(len(cands)))
        prompt, _ = render(r, order)
        raw = ask(r["image_file"], prompt)
        m = re.search(r"\b([a-h])\b", raw.lower())
        j = string.ascii_lowercase.index(m.group(1)) if m else None
        pred = cands[order[j]] if (j is not None and j < len(cands)) else None
        if m: letters[m.group(1)] += 1
    ok = dim_ok(pred, float(r["gold_lo"]), float(r["gold_hi"]), r["gold_unit"])
    out.append({**{k: r[k] for k in ("image_file","attribute","gold_lo","gold_hi","gold_unit","gold_index")},
                "n_cands": len(cands), "raw": raw,
                "pred": list(pred) if pred else None, "dim": bool(ok)})
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(eval_rows)}  dim {100*sum(x['dim'] for x in out)/len(out):.1f}%", flush=True)

json.dump(out, open(f"{WORK}/vlm_lora_results.json","w"), indent=1)

def wilson(k, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return ((c-h)/d, (c+h)/d)

n = len(out); k = sum(x["dim"] for x in out)
lo, hi = wilson(k, n)
print(f"\n{'system':<34}{'n':>5}{'dim':>8}{'95% CI':>16}")
print("-"*63)
print(f"{'LoRA fine-tuned (selection)':<34}{n:>5}{100*k/n:>7.1f}%{f'[{100*lo:.1f}, {100*hi:.1f}]':>16}")
print(f"{'zero-shot select':<34}{398:>5}{34.2:>7.1f}%")
print(f"{'zero-shot forced':<34}{398:>5}{35.4:>7.1f}%")
print(f"{'tagger + rules cascade':<34}{'':>5}{36.2:>7.1f}%")
print(f"{'gold among candidates (ceiling)':<34}{'':>5}{70.1:>7.1f}%")

multi = [x for x in out if x["n_cands"] >= 2]
gp = [x for x in multi if x["gold_index"] >= 0]
km = sum(x["dim"] for x in gp)
lo2, hi2 = wilson(km, len(gp))
print(f"\ngold-present, 2+ candidates: {100*km/len(gp):.1f}% "
      f"[{100*lo2:.1f}, {100*hi2:.1f}]  (zero-shot 43.3%, random 33.7%)")

tot = sum(letters.values()) or 1
print("\nletter distribution (was b 36.9% / c 35.4% / a 11.1% zero-shot):")
print("  " + "   ".join(f"({L}) {100*c/tot:.1f}%" for L, c in sorted(letters.items())))
print("\nDownload vlm_lora_results.json and the adapter folder from Output.")